# Filter Alerts by FIRMS Fire Detection

This notebook filters deforestation alerts by matching them with MODIS FIRMS (Fire Information for Resource Management System) fire detections.

## Purpose
- Load alert data from Google Earth Engine
- Match alerts with MODIS fire detections based on spatial and temporal proximity
- Return a filtered `ee.Image` that can be used with the Alert Navigator UI

## Output
The filtered image can be passed to `AlertGroupingNavigator` instead of an asset ID.

## 1. Import Libraries

In [ ]:
import ee
import sys
sys.path.append('..')
from scripts.helpers import decimal_year_to_ee_date
from scripts.helpers import date_to_decimal_year
ee.Initialize()

## 2. Configuration Parameters

In [ ]:
# ========== CONFIGURATION PARAMETERS ==========

ALERT_ASSET_ID = "projects/ee-dfgm2006/assets/fires_colombia/Change_Alerts_CCDC_SABANAS_FIRE_landsatOnly_2019_2025_aggresiveMask_sf"

FOREST_ASSET_ID = "projects/ee-dfgm2006/assets/fires_colombia/cambio2017-2018"

AOI = ee.Image(FOREST_ASSET_ID).geometry()  # Using forest mask geometry as AOI


# FIRMS filtering parameters
DELTA_DAYS = 3  # Temporal window: ±3 days for matching alerts with fire detections
SCALE_FIRMS = 1000  # MODIS resolution in meters

print("Configuration set:")
print(f"  Alert Asset: {ALERT_ASSET_ID}")
print(f"  AOI defined")
print(f"  Temporal window: ±{DELTA_DAYS} days")

## 3. Helper Functions

Date conversion functions for working with decimal year format.

## 4. Load Alert Data and Determine Time Range

In [ ]:
# Load alerts


alerts = ee.Image(ALERT_ASSET_ID)

# Optional: Apply forest mask
# forest = ee.Image(FOREST_ASSET_ID)
# fnf = forest.select("b1").eq(1).selfMask()
# alerts = alerts.updateMask(fnf)

print("Alert bands:", alerts.bandNames().getInfo())

# Get confirmation_date band
conf_date = alerts.select('confirmation_date')
conf_mask = conf_date.gt(0)

# Get time range from confirmation dates
conf_stats = conf_date.updateMask(conf_mask).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=AOI,
    scale=500,  # Coarse scale for speed
    maxPixels=1e7
)

print("\nConfirmation date statistics (decimal years):")
print(conf_stats.getInfo())

# Extract min/max decimal years
min_dec = ee.Number(conf_stats.get('confirmation_date_min'))
max_dec = ee.Number(conf_stats.get('confirmation_date_max'))

# Convert to dates with buffer
start_date = decimal_year_to_ee_date(min_dec).advance(-7, 'day')
end_date = decimal_year_to_ee_date(max_dec).advance(7, 'day')

print(f"\nFIRMS time window:")
print(f"  Start: {start_date.format('YYYY-MM-dd').getInfo()}")
print(f"  End: {end_date.format('YYYY-MM-dd').getInfo()}")

## 5. Prepare Confirmation Date at FIRMS Resolution

In [ ]:
# Clip confirmation date to AOI
conf_date_1km = conf_date.clip(AOI)
conf_mask_1km = conf_mask.clip(AOI)

print(f"✓ Confirmation date prepared at {SCALE_FIRMS}m resolution")

## 6. Build Fire-Alert Mask from MODIS FIRMS

Match MODIS fire detections with alert dates within the temporal window.

In [ ]:
# Calculate temporal window in decimal years
delta_year = ee.Number(DELTA_DAYS).divide(365)

# Load MODIS FIRMS collection
modis_ic = ee.ImageCollection('FIRMS') \
    .filterBounds(AOI) \
    .filterDate(start_date, end_date)

print(f"Number of MODIS FIRMS images: {modis_ic.size().getInfo()}")

# Map over each FIRMS image to create fire-alert masks
def create_fire_alert_mask(img):
    """
    For each MODIS FIRMS image:
    1. Identify fire pixels (T21 > 0)
    2. Check temporal match with confirmation_date
    3. Return spatio-temporal match
    """
    # fire pixels: T21 > 0
    fire = img.select('T21').gt(0).selfMask().clip(AOI)
    
    # temporal match with confirmation_date
    img_date = ee.Date(img.get('system:time_start'))
    img_dec_year = date_to_decimal_year(img_date)
    
    dy_img = ee.Image.constant(img_dec_year).toFloat().clip(AOI)
    
    temporal_match = conf_date_1km \
        .subtract(dy_img) \
        .abs() \
        .lte(delta_year) \
        .updateMask(conf_mask_1km)
    
    # spatio-temporal match = fire AND temporal_match
    fire_alert = fire.And(temporal_match)
    
    return fire_alert.rename('fire_alert') \
        .copyProperties(img, ['system:time_start'])

fire_alert_ic = modis_ic.map(create_fire_alert_mask)
fire_alert_1km = fire_alert_ic.max().rename('fire_alert_1km').clip(AOI)

## 7. Apply FIRMS Filter to Alerts

Create the final filtered alert image.

In [ ]:
# Apply the fire-alert mask to the original alerts
alerts_filtered_by_firms = alerts.updateMask(fire_alert_1km)


## 8. Export Filtered Alerts to Earth Engine Asset

In [ ]:
# Export configuration
EXPORT_ASSET_ID = ALERT_ASSET_ID + "_filtered_by_firms"
EXPORT_DESCRIPTION = "alerts_filtered_by_firms"

# Create export task
export_task = ee.batch.Export.image.toAsset(
    image=alerts_filtered_by_firms,
    description=EXPORT_DESCRIPTION,
    assetId=EXPORT_ASSET_ID,
    region=AOI,
    scale=30,  # Native resolution
    maxPixels=1e10,
    pyramidingPolicy={
        '.default': 'sample',
        'confirmation_date': 'sample',
        'difference': 'sample'
    }
)

# Start the export
export_task.start()

print("✓ Export task started!")
print(f"  Description: {EXPORT_DESCRIPTION}")
print(f"  Asset ID: {EXPORT_ASSET_ID}")
print(f"  Region: AOI")
print(f"  Scale: 30m")
print("\nMonitor the task at: https://code.earthengine.google.com/tasks")
print("Or check status with: ee.batch.Task.list()")
